In [ ]:
import pandas as pd

df = pd.read_csv('iris.csv')
df.head(5)


In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
# 種類列にどのような値があるか確認(重複を考慮してカウント)
species = df['種類'].unique()
species

In [ ]:
species.size

In [ ]:
# 指定した列のデータ個数の集計
df['種類'].value_counts()

In [ ]:
df.tail(5)

In [ ]:
import matplotlib.pyplot as plt
# Notebook でグラフをインライン表示
%matplotlib inline

# 散布図を作成
plt.figure(figsize=(8, 6))
 # 種類ごとに色分けして描画
species_list = df["種類"].unique()
 
for species in species_list:
    subset = df[df["種類"] == species]
    plt.scatter(
        subset["花弁幅"],
        subset["花弁長さ"],
        label=species )

# 軸ラベル・タイトル・凡例
plt.xlabel("Petal Width")
plt.ylabel("Petal Length")
plt.title("Iris Petal Width vs Length")
plt.legend()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
features = ['がく片長さ','がく片幅','花弁長さ','花弁幅']

sns.pairplot(df[features + ["種類"]], hue="種類")
plt.show()
plt.close

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
features = ['がく片長さ','がく片幅','花弁長さ','花弁幅']
df_melt = df.melt(id_vars='種類', value_vars=features, var_name='feature', value_name='value')

# 箱ひげ図を描画
plt.figure(figsize=(10,6))
sns.boxplot(data=df_melt, x='feature', y='value', hue='種類')
plt.title('Iris Dataset Boxplot')
plt.xlabel('Feature')
plt.ylabel('Value')
plt.legend(title='Species')
plt.show()
plt.close

In [ ]:
# 欠損値の確認
df.isnull()

In [ ]:
# anyメソッドで列単位で欠損値の有無を確認
df.isnull().any(axis=0)

In [ ]:
# 各列の合計値を求める
df.sum()

In [ ]:
# 各列の欠損値の数を調べる
df.isnull().sum()

In [ ]:
# 欠損値を含む行または列の削除
df2 = df.dropna(how='any',axis=0)
df2.tail(5)

In [ ]:
df2.isnull()

In [ ]:
# 元のデータの欠損値の有無の確認
df.isnull().any(axis=0)

In [ ]:
# 新しいデータの欠損値の有無の確認
df2.isnull().any(axis=0)

In [ ]:
# 欠損値の穴埋め
df['花弁長さ']= df['花弁長さ'].fillna(0)
df.tail(5)

In [ ]:
# 穴埋めしたデータの欠損値の有無の確認
df.isnull().any(axis=0)

In [ ]:
# 平均値の計算
df.mean(numeric_only=True)

In [ ]:
# 特定の列の平均値の計算
df['がく片長さ'].mean()

In [ ]:
# 標準偏差の計算
df.std(numeric_only=True)

In [ ]:
# 平均値を求めてデータフレームの欠損値と置き換える
df = pd.read_csv('iris.csv')
colmean = df.mean(numeric_only=True)
colmean
df2 = df.fillna(colmean)
df2.tail(5)

In [ ]:
df2.isnull().any(axis=0)

### 5.2.7 特徴量と正解データの取り出し

In [ ]:
# 特徴量と正解データを変数に代入
xcol = ['がく片長さ','がく片幅','花弁長さ','花弁幅']

x = df2[xcol]
t = df2['種類']
x,t

### 5.3.3 モデルの作成

In [ ]:
# モジュールのインポート
from sklearn import tree
# モデルの作成
# DecisionTreeClassifier(criterion,　splitter)
#  criterion : gini=ジニ不純度(既定), entropy=情報利得), log_loss=ログ損失(分類の確率的評価))
#　splitter ： best=最良の分割を探索, random=ランダムに候補を選び、その中で最良を選択（高速化向け）
model = tree.DecisionTreeClassifier(max_depth=2, random_state=0)
# model = tree.DecisionTreeClassifier(max_depth=3, random_state=0)
# model = tree.DecisionTreeClassifier(splitter='best')
# モデルの学習
model.fit(x,t)
# モデルの正解率の計算
model.score(x,t)

In [ ]:
# ホールドアウト法により訓練データとテストデータに分割する
from sklearn.model_selection import train_test_split

# x_train, y_train：学習に利用する訓練データ
# x_test, y_test：検証に利用するテストデータ
# test_size　: 検証に利用する割合、注意：整数値を指定するとテストデータ件数とみなされる
x_train, x_test, y_train, y_test = train_test_split(x,t,test_size=0.3, random_state=0)
print("学習用訓練データの特徴量の行列サイズ：",x_train.shape)
print(x_train.head(3), "\n")
print("検証用テストデータの特徴量の行列サイズ：",x_test.shape)
print(x_test.head(3),"\n")
print("学習用訓練データの正解の行列サイズ：",y_train.shape)
print(y_train.head(3), "\n")
print("検証用テストデータの正解の行列サイズ：",y_test.shape)
print(y_test.head(3),"\n")

In [ ]:
#　ホールドアウト法により分割した7割の訓練データで学習
model.fit(x_train,y_train)
#　ホールドアウト法により分割した3割のテストデータで正解率を計算
model.score(x_test,y_test)

In [ ]:
# モデルの保存
import pickle
with open('irismodel.pkl','wb') as f:
    pickle.dump(model,f)

In [ ]:
# 分岐条件の列を表示する
model.tree_.feature

In [ ]:
# 分岐条件の閾値を含む配列を表示する
model.tree_.threshold

In [ ]:
# リーフ(葉)に到達したデータの数を返す
print("各ノードに到達した数",model.tree_.weighted_n_node_samples)
print("アヤメの種類",model.classes_)
idx = [1,3,4]
for i in idx:
    print("ノード番号",i,"に到達した割合",model.tree_.value[i]*model.tree_.weighted_n_node_samples[i])



In [ ]:
# plot_tree関数で決定木を描画する
x_train.columns = ['gaku_nagasa','gaku_haba', 'kaben_nagasa', 'kaben_haba']

from sklearn.tree import plot_tree
plot_tree(model, feature_names=x_train.columns,filled=True)